In [1]:
from pathlib import Path
import sys
import time

import pandas as pd
from IPython.display import display

_directorios = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
RUTA_PROYECTO = next(
    (directorio for directorio in _directorios if (directorio / "modelo_calendario.py").is_file()),
    None,
)
if RUTA_PROYECTO is None:
    raise FileNotFoundError("No se encontró modelo_calendario.py en el proyecto ni sus carpetas padre.")

sys.path.insert(0, str(RUTA_PROYECTO))
from modelo_calendario import resolver_instancia

TEMPORADA = "2025-2026"
URL_CSV = str((Path.cwd().resolve().parent / "BASES" / "2025-2026.csv"))

PARAMETROS_MODELO = {
    "clasicos": [],
    "estadios_compartidos": [],
    "equipos_europeos": [],
    "jornadas_criticas": [],
    "max_clasicos_por_jornada": 1,
    "peso_breaks": 1.0,
    "peso_viajes": 1.0,
    "numero_equipos_esperado": 20,
}

configuraciones = {
    "Normal (Gurobi por defecto)": False,
    "Sin cortes ni heurísticas": True,
}

TIEMPO_TOTAL = 3600
MARGEN_SEGUNDOS = 30
inicio = time.monotonic()
resultados = {}

for indice, (nombre, desactivar_cortes_heuristicas) in enumerate(configuraciones.items()):
    pendientes = len(configuraciones) - indice
    disponible = TIEMPO_TOTAL - (time.monotonic() - inicio) - MARGEN_SEGUNDOS

    if disponible < 1:
        print(f"Se omite {nombre}: se agotó el presupuesto de tiempo.")
        break

    tiempo_limite = max(1, int(disponible / pendientes))
    print(f"{nombre}: límite de Gurobi de {tiempo_limite} segundos.")

    resultados[nombre] = resolver_instancia(
        TEMPORADA,
        URL_CSV,
        desactivar_cortes_heuristicas=desactivar_cortes_heuristicas,
        tiempo_limite=tiempo_limite,
        **PARAMETROS_MODELO,
    )

reporte = pd.DataFrame(
    [
        {
            "Configuración": nombre,
            "Estado Gurobi": resultado["estado"],
            "Objetivo (ObjVal)": resultado["objetivo"],
            "Cota (ObjBoundC)": resultado["cota"],
            "Gap (MIPGap)": resultado["gap"],
            "Tiempo (s)": resultado["tiempo"],
            "Nodos (NodeCount)": resultado["nodos"],
        }
        for nombre, resultado in resultados.items()
    ]
)
reporte["Objetivo (ObjVal)"] = reporte["Objetivo (ObjVal)"].fillna("No factible")
reporte["Gap (MIPGap)"] = reporte["Gap (MIPGap)"].fillna("No factible")
estado_map = {
    1: "OPTIMAL",
    2: "INFEASIBLE",
    3: "INF_OR_UNBD",
    4: "CUTOFF",
    5: "LOADED",
    6: "UNKNOWN",
}
reporte["Estado Gurobi"] = reporte["Estado Gurobi"].map(lambda v: estado_map.get(v, v))
display(reporte)

if "Normal (Gurobi por defecto)" in resultados:
    resultado_normal = resultados["Normal (Gurobi por defecto)"]
    print(f"Partidos históricos descargados: {len(resultado_normal['partidos_historicos'])}")
    print(f"Breaks: {resultado_normal['breaks']} | Viajes críticos: {resultado_normal['viajes_criticos']}")
    display(resultado_normal["calendario"])
    display(resultado_normal["partidos_historicos"][["Date", "HomeTeam", "AwayTeam"]].head(10))

Normal (Gurobi por defecto): límite de Gurobi de 1784 segundos.
Set parameter Username
Set parameter LicenseID to value 2834871
Academic license - for non-commercial use only - expires 2027-06-12
Set parameter TimeLimit to value 1784
Set parameter OutputFlag to value 1
2025-2026: 20 equipos, 38 jornadas, 16,680 variables y 7,080 restricciones.
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  1784

Optimize a model with 7080 rows, 16680 columns and 71800 nonzeros (Min)
Model fingerprint: 0x1921d8ff
Model has 1480 linear objective coefficients
Variable types: 1480 continuous, 15200 integer (15200 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01

,Configuración,Estado Gurobi,Objetivo (ObjVal),Cota (ObjBoundC),Gap (MIPGap),Tiempo (s),Nodos (NodeCount)
0,Normal (Gurobi por defecto),9,102.0,5.142857,0.941176,8775.226,110.0


Partidos históricos descargados: 380
Breaks: 102.0 | Viajes críticos: 0


,Jornada,Local,Visitante
0,1,Arsenal,Burnley
1,1,Aston Villa,West Ham
2,1,Brentford,Sunderland
3,1,Fulham,Bournemouth
4,1,Liverpool,Chelsea
...,...,...,...
375,38,Leeds,Sunderland
376,38,Man United,Liverpool
377,38,Nott'm Forest,Newcastle
378,38,Tottenham,Man City


,Date,HomeTeam,AwayTeam
0,15/08/2025,Liverpool,Bournemouth
1,16/08/2025,Aston Villa,Newcastle
2,16/08/2025,Brighton,Fulham
3,16/08/2025,Sunderland,West Ham
4,16/08/2025,Tottenham,Burnley
5,16/08/2025,Wolves,Man City
6,17/08/2025,Chelsea,Crystal Palace
7,17/08/2025,Nott'm Forest,Brentford
8,17/08/2025,Man United,Arsenal
9,18/08/2025,Leeds,Everton
